# jobs

> a durable queue in the vault file: retries with backoff, leases that survive a crash, and a dead letter


In [ ]:
#| default_exp jobs

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

One table, one connection, no daemon. `enqueue` writes a row; `claim` takes it with an
`UPDATE ... RETURNING` through a partial index, so two workers racing get disjoint sets; a worker
that dies holds a lease that expires, and `reclaim` puts the job back. Failures back off
exponentially and dead-letter once the attempts are spent.

The queue lives in the same SQLite file as the documents, so enqueueing inside a `write_txn`
commits with the write that caused it. Nothing depends on the vault: `Queue` takes a database and a
dict of handlers, and [`acquire`](01_acquire.ipynb) is what fills them in.

In [ ]:
#| export
import json, random, time, uuid
from fastcore.all import AttrDict, L, first, patch, store_attr
from litesearch import write_txn

STATES = ('ready', 'running', 'done', 'dead')

In [ ]:
#| export
class Retry(Exception):
    'Raise from a handler to retry work that did not raise on its own: a bot wall, a 429, an empty read.'
    def __init__(self, msg:str='', after:float=None):
        super().__init__(msg)
        self.after = after   # seconds until the retry; None uses the backoff schedule

def backoff(attempt:int,       # failures so far; 1 on the first
            base:float=30,     # first delay, seconds
            cap:float=21600,   # longest delay, 6h
            jitter:float=0.25  # +/- fraction, so jobs that failed together do not retry together
) -> float:
    'Exponential delay before the next attempt.'
    return max(1., min(cap, base * 2**(attempt-1)) * (1 + random.uniform(-jitter, jitter)))

In [ ]:
#| hide
test_eq(backoff(1, base=10, jitter=0) , 10)
test_eq(backoff(4, base=10, jitter=0) , 80)
test_eq(backoff(99, base=10, jitter=0), 21600)   # capped, not astronomical
assert 7.5 <= backoff(1, base=10) <= 12.5

## The queue

`payload` is JSON in, dict out. `key` is what makes an enqueue idempotent: re-running the same
fan-out does not double the work, because a key already `ready` or `running` is left alone.

In [ ]:
#| export
def _row(r) -> AttrDict:
    'A job row with its payload parsed.'
    d = dict(r)
    p = d.get('payload')
    d['payload'] = json.loads(p) if isinstance(p, str) else (p or {})
    return AttrDict(d)

class Queue:
    "An at-least-once job queue inside an open litesearch database."
    def __init__(self,
                 db,                     # open litesearch/fastlite Database; the queue lives in its file
                 lease:float=600,        # seconds a claim is held before another worker may take the job
                 max_attempts:int=5,     # attempts before a job is dead-lettered
                 base:float=30,          # first retry delay; doubles per attempt
                 keep:float=604800):     # how long finished jobs and run history survive `purge`, 7d
        store_attr()
        self.handlers = {}
        self._mk()

    def _mk(self):
        'The two tables and their indexes, created on first use.'
        self.t = self.db.t.jobs
        self.t.create(id=str, kind=str, payload=str, key=str, state=str, priority=int, run_at=float,
                      lease_until=float, worker=str, attempt=int, max_attempts=int, error=str,
                      created_at=float, updated_at=float, pk='id', if_not_exists=True)
        self.runs = self.db.t.job_runs
        self.runs.create(id=str, job_id=str, kind=str, worker=str, status=str, error=str,
                         started_at=float, ended_at=float, took=float, pk='id', if_not_exists=True)
        for sql in (
            "CREATE INDEX IF NOT EXISTS jobs_due ON jobs(priority, run_at) WHERE state='ready'",
            "CREATE INDEX IF NOT EXISTS jobs_lease ON jobs(lease_until) WHERE state='running'",
            "CREATE UNIQUE INDEX IF NOT EXISTS jobs_key ON jobs(kind, key) "
            "WHERE key IS NOT NULL AND state IN ('ready','running')",
            "CREATE INDEX IF NOT EXISTS job_runs_job ON job_runs(job_id, started_at)"):
            self.db.conn.execute(sql)

    def __repr__(self):
        s = self.stats()
        return (f"Queue({s['ready']} ready, {s['running']} running, {s['done']} done, {s['dead']} dead)")

In [ ]:
#| export
@patch
def enqueue(self:Queue,
            kind:str,             # which handler runs it
            payload:dict=None,    # arguments, stored as JSON
            run_at:float=None,    # epoch seconds; defaults to now
            priority:int=0,       # lower runs first
            key:str=None,         # idempotency key: an identical key still ready or running wins
            max_attempts:int=None
) -> AttrDict:
    'Add one job, or return the pending one that `key` already names.'
    now = time.time()
    row = dict(id=uuid.uuid4().hex[:12], kind=kind, payload=json.dumps(payload or {}), key=key,
               state='ready', priority=priority, run_at=run_at if run_at is not None else now,
               lease_until=0., worker=None, attempt=0, max_attempts=max_attempts or self.max_attempts,
               error=None, created_at=now, updated_at=now)
    with write_txn(self.db):
        if key is not None and (was := first(self.db.q(
                "SELECT * FROM jobs WHERE kind=? AND key=? AND state IN ('ready','running')", [kind, key]))):
            return _row(was)
        self.t.insert(row)
    return _row(row)

@patch
def enqueue_all(self:Queue, kind:str, payloads, **kw) -> L:
    'Enqueue many jobs of one kind in a single transaction.'
    with write_txn(self.db): return L(payloads).map(lambda p: self.enqueue(kind, p, **kw))

### Claiming

The claim is one statement. SQLite holds the write lock for its duration, so the `SELECT` that
picks the rows and the `UPDATE` that marks them cannot interleave with another worker's. What a
worker gets back is already `running` and already leased.

In [ ]:
#| export
@patch
def claim(self:Queue,
          worker:str='worker',  # who is holding the lease; recorded on the row and in history
          n:int=1,              # how many jobs to take
          now:float=None
) -> L:
    'Take up to `n` due jobs, atomically. Concurrent callers get disjoint sets.'
    now = time.time() if now is None else now
    with write_txn(self.db):
        rows = self.db.q(
            'UPDATE jobs SET state=?, worker=?, lease_until=?, updated_at=? WHERE id IN '
            '(SELECT id FROM jobs WHERE state=? AND run_at<=? ORDER BY priority, run_at LIMIT ?) '
            'RETURNING *', ['running', worker, now + self.lease, now, 'ready', now, n])
    # ORDER BY picks *which* rows; RETURNING hands them back in no defined order, so sort again here
    return L(sorted(rows, key=lambda r: (r['priority'], r['run_at']))).map(_row)

@patch
def ack(self:Queue, job) -> AttrDict:
    'Mark a claimed job finished.'
    self.db.q('UPDATE jobs SET state=?, error=NULL, worker=NULL, lease_until=0, updated_at=? WHERE id=?',
              ['done', time.time(), job['id']])
    return _row(dict(job, state='done'))

@patch
def fail(self:Queue,
         job,                # the claimed job
         error:str,          # what went wrong; truncated to 500 chars on the row
         after:float=None    # seconds until the retry; None uses `backoff`
) -> AttrDict:
    'Back off and retry, or dead-letter once the attempts are spent.'
    now, err = time.time(), str(error)[:500]
    with write_txn(self.db):
        cur = first(self.db.q('SELECT * FROM jobs WHERE id=?', [job['id']]))
        if cur is None: return _row(dict(job, state='gone'))
        att = cur['attempt'] + 1
        dead = att >= (cur['max_attempts'] or self.max_attempts)
        nxt = cur['run_at'] if dead else now + (backoff(att, base=self.base) if after is None else after)
        self.db.q('UPDATE jobs SET state=?, attempt=?, run_at=?, error=?, worker=NULL, lease_until=0, '
                  'updated_at=? WHERE id=?', ['dead' if dead else 'ready', att, nxt, err, now, cur['id']])
    return _row(dict(cur, state='dead' if dead else 'ready', attempt=att, run_at=nxt, error=err))

@patch
def reclaim(self:Queue, now:float=None) -> L:
    'Return jobs whose lease expired to the queue: a worker killed mid-job loses its claim, not its job.'
    now = time.time() if now is None else now
    exp = self.db.q('SELECT * FROM jobs WHERE state=? AND lease_until>0 AND lease_until<=?', ['running', now])
    return L(exp).map(lambda j: self.fail(j, 'lease expired'))

### Running

A handler is `fn(payload)`. Returning a dict with `skipped` records a skip and does not retry, which
is what the acquisition methods already return when a page has no transcript or is not a PDF. Raise
`Retry` for the opposite case, where nothing threw but the work should happen again.

In [ ]:
#| export
@patch
def on(self:Queue, kind:str):
    'Decorator registering the handler for `kind`.'
    def _f(fn):
        self.handlers[kind] = fn
        return fn
    return _f

@patch
def register(self:Queue, kind:str, fn):
    'Register the handler for `kind`.'
    self.handlers[kind] = fn
    return fn

@patch
def _ran(self:Queue, job, status:str, t0:float, error:str=None, result=None) -> AttrDict:
    'Write one row of run history and return what the caller reports.'
    now = time.time()
    self.runs.insert(dict(id=uuid.uuid4().hex[:12], job_id=job['id'], kind=job['kind'],
                          worker=job.get('worker'), status=status, error=(error or None),
                          started_at=t0, ended_at=now, took=round(now-t0, 3)))
    return AttrDict(job_id=job['id'], kind=job['kind'], status=status, took=round(now-t0, 3),
                    error=error, result=result)

@patch
def run_one(self:Queue, job) -> AttrDict:
    'Dispatch one claimed job, then ack or fail it, and record the run either way.'
    t0 = time.time()
    if (fn := self.handlers.get(job['kind'])) is None:
        err = f"no handler for {job['kind']!r}"
        return self._ran(self.fail(job, err), 'error', t0, err)
    try: res = fn(job['payload'])
    except Retry as e:
        return self._ran(self.fail(job, f'Retry: {e}', after=e.after), 'retry', t0, str(e))
    except Exception as e:
        err = f'{type(e).__name__}: {str(e)[:200]}'
        return self._ran(self.fail(job, err), 'error', t0, err)
    self.ack(job)
    return self._ran(job, 'skipped' if isinstance(res, dict) and res.get('skipped') else 'ok',
                     t0, None, res)

@patch
def drain(self:Queue,
          worker:str='worker',  # lease holder recorded on each job
          limit:int=1000,       # jobs to run before returning; a ceiling, not a target
          batch:int=1,          # jobs claimed per round trip
          now:float=None
) -> L:
    'Claim and run until the queue is empty or `limit` jobs have run. This is the tick a cron calls.'
    out = L()
    while len(out) < limit:
        got = self.claim(worker, n=min(batch, limit-len(out)), now=now)
        if not got: break
        out += got.map(self.run_one)
    return out

### Looking at it

`dead` is the queue's inbox for a human. Nothing is deleted on failure, so `retry` is a state flip
and the payload that failed is still there to read.

In [ ]:
#| export
@patch
def jobs(self:Queue, state:str=None, kind:str=None, limit:int=100) -> L:
    'Jobs, soonest first, filtered by state and kind.'
    w, a = [], []
    if state: w.append('state=?'); a.append(state)
    if kind:  w.append('kind=?');  a.append(kind)
    sql = 'SELECT * FROM jobs' + (' WHERE ' + ' AND '.join(w) if w else '') + ' ORDER BY run_at LIMIT ?'
    return L(self.db.q(sql, a + [limit])).map(_row)

@patch
def dead(self:Queue, limit:int=100) -> L:
    'Jobs that used up their attempts, with the error that finished them off.'
    return self.jobs(state='dead', limit=limit)

@patch
def retry(self:Queue, job_id:str, after:float=0) -> AttrDict:
    'Put a dead or pending job back on the queue now, with its attempt count reset.'
    self.db.q('UPDATE jobs SET state=?, attempt=0, run_at=?, error=NULL, worker=NULL, lease_until=0, '
              'updated_at=? WHERE id=?', ['ready', time.time()+after, time.time(), job_id])
    return first(self.jobs(limit=1) and L(self.db.q('SELECT * FROM jobs WHERE id=?', [job_id])).map(_row))

@patch
def history(self:Queue, job_id:str=None, limit:int=50) -> L:
    'Run history, newest first.'
    if job_id: return L(self.db.q('SELECT * FROM job_runs WHERE job_id=? ORDER BY started_at DESC LIMIT ?',
                                  [job_id, limit]))
    return L(self.db.q('SELECT * FROM job_runs ORDER BY started_at DESC LIMIT ?', [limit]))

@patch
def stats(self:Queue) -> dict:
    'How many jobs in each state, and when the next one is due.'
    by = {r['state']: r['n'] for r in self.db.q('SELECT state, count(*) n FROM jobs GROUP BY state')}
    nxt = first(self.db.q("SELECT min(run_at) t FROM jobs WHERE state='ready'"))
    return dict({s: by.get(s, 0) for s in STATES}, next_due=(nxt or {}).get('t'))

@patch
def purge(self:Queue, before:float=None) -> dict:
    'Delete finished jobs and run history older than `keep`. Dead jobs stay.'
    cut = (time.time() - self.keep) if before is None else before
    d = self.db.q("DELETE FROM jobs WHERE state='done' AND updated_at<? RETURNING id", [cut])
    r = self.db.q('DELETE FROM job_runs WHERE ended_at<? RETURNING id', [cut])
    return dict(jobs=len(d), runs=len(r))

## Try it

In [ ]:
from litesearch import database

db = database()
q  = Queue(db, lease=60, max_attempts=3, base=1)
seen = []
q.register('echo', lambda p: seen.append(p['n']) or dict(n=p['n']))
q.enqueue_all('echo', [dict(n=i) for i in range(3)])
q.drain('w1')
seen, q.stats()

In [ ]:
#| hide
# claim is atomic: two workers over one ready job get it once between them, never twice
db2 = database()
q2  = Queue(db2, lease=60, max_attempts=3, base=1)
q2.enqueue('t', dict(a=1))
a, b = q2.claim('w1', n=5), q2.claim('w2', n=5)
test_eq(len(a) + len(b), 1)
test_eq(first(a or b)['payload'], dict(a=1))
test_eq(first(a or b)['state'], 'running')
# ...and a claimed job is no longer due, so a third worker sees nothing
test_eq(q2.claim('w3', n=5), [])

In [ ]:
#| hide
# a failure backs off and comes back; the third one is dead, and the payload survives to be read
q3 = Queue(database(), lease=60, max_attempts=3, base=100)
j  = q3.enqueue('t', dict(url='http://x'))
for i in (1, 2):
    c = first(q3.claim('w'))
    out = q3.fail(c, f'boom {i}')
    test_eq(out['state'], 'ready')
    test_eq(out['attempt'], i)
    assert out['run_at'] > time.time(), 'a retry is scheduled in the future'
    q3.db.q("UPDATE jobs SET run_at=0 WHERE id=?", [j['id']])   # skip the wait
c = first(q3.claim('w'))
test_eq(q3.fail(c, 'boom 3')['state'], 'dead')
test_eq(len(q3.dead()), 1)
test_eq(q3.dead()[0]['payload'], dict(url='http://x'))
test_eq(q3.dead()[0]['error'], 'boom 3')
test_eq(q3.claim('w'), [])                     # dead jobs are not due
test_eq(q3.retry(j['id'])['state'], 'ready')   # ...until a human puts one back
test_eq(len(first(q3.claim('w')) and [1]), 1)

In [ ]:
#| hide
# a worker killed mid-job never acks; the lease expires and the job comes back
q4 = Queue(database(), lease=30, max_attempts=3, base=1)
q4.enqueue('t', dict(a=1))
t = time.time()
test_eq(len(q4.claim('doomed', now=t)), 1)
test_eq(q4.reclaim(now=t+10), [])              # lease still held
back = q4.reclaim(now=t+31)
test_eq(len(back), 1)
test_eq(back[0]['state'], 'ready')
test_eq(back[0]['attempt'], 1)                 # a job that kills its worker every time still dies
test_eq(back[0]['error'], 'lease expired')

In [ ]:
#| hide
# `key` is what makes a re-run of a fan-out cheap: the same key pending twice is one job
q5 = Queue(database(), base=1)
a = q5.enqueue('file', dict(p='/x'), key='/x')
b = q5.enqueue('file', dict(p='/x'), key='/x')
test_eq(a['id'], b['id'])
test_eq(len(q5.jobs()), 1)
q5.register('file', lambda p: dict(ok=1))
q5.drain('w')
c = q5.enqueue('file', dict(p='/x'), key='/x')  # done, so the same work can be asked for again
test_ne(c['id'], a['id'])
test_eq(len(q5.jobs()), 2)

In [ ]:
#| hide
# the three handler outcomes: ok, a skip that is final, and a Retry that is not
q6 = Queue(database(), max_attempts=3, base=1)
q6.register('ok',   lambda p: dict(added=1))
q6.register('skip', lambda p: dict(skipped='no transcript'))
q6.register('wall', lambda p: (_ for _ in ()).throw(Retry('bot wall', after=5)))
q6.register('bad',  lambda p: 1/0)
for k in ('ok', 'skip', 'wall', 'bad'): q6.enqueue(k)
r = {x['kind']: x for x in q6.drain('w')}
test_eq(r['ok']['status'], 'ok')
test_eq(r['skip']['status'], 'skipped')
test_eq(r['wall']['status'], 'retry')
test_eq(r['bad']['status'], 'error')
assert 'ZeroDivisionError' in r['bad']['error']
test_eq({j['kind'] for j in q6.jobs(state='done')}, {'ok', 'skip'})   # a skip is not a failure
test_eq({j['kind'] for j in q6.jobs(state='ready')}, {'wall', 'bad'})
test_eq(len(q6.history()), 4)
# an unregistered kind fails like any other error rather than disappearing
q6.enqueue('nobody')
test_eq(first(q6.drain('w', limit=1))['status'], 'error')

In [ ]:
#| hide
# priority orders the queue, and a job scheduled ahead is not claimed early
q7 = Queue(database(), base=1)
t = time.time()
q7.enqueue('t', dict(n='later'),  priority=0,  run_at=t+3600)
q7.enqueue('t', dict(n='low'),    priority=5,  run_at=t)
q7.enqueue('t', dict(n='urgent'), priority=-1, run_at=t)
test_eq([j['payload']['n'] for j in q7.claim('w', n=9, now=t)], ['urgent', 'low'])
test_eq([j['payload']['n'] for j in q7.claim('w', n=9, now=t+3601)], ['later'])

In [ ]:
#| hide
# purge clears finished work and its history, and leaves the dead letter alone
q8 = Queue(database(), max_attempts=1, base=1)
q8.register('ok', lambda p: dict(ok=1))
q8.enqueue('ok'); q8.enqueue('nope')
q8.drain('w')
test_eq(q8.stats()['done'], 1)
test_eq(q8.stats()['dead'], 1)
test_eq(q8.purge(before=time.time()+1), dict(jobs=1, runs=2))
test_eq(q8.stats(), dict(ready=0, running=0, done=0, dead=1, next_due=None))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()